# Dataset

Bloc C — génération des réponses d'apprenant étiquetées (PARFAIT, PARTIEL, INCORRECT) à partir des fiches concept.

Les cellules s'exécutent dans l'ordre. Le dossier `json` des fiches doit être à côté du notebook.

In [ ]:
import json
from pathlib import Path

# Chemin relatif : le script se lance depuis la racine du dépôt.
FICHES_FOLDER = "json"
REQUIRED_KEYS = ["id", "concept", "explication", "questions"]
VERDICTS = ["PARFAIT", "PARTIEL", "INCORRECT"]
# Le LLM de Colab n'a pas de réglage de température : on réessaie quand sa sortie est inutilisable.
MAX_ATTEMPTS = 3
# Règles de verdict de la US2 (backlog), rappelées au LLM qui écrit les réponses d'apprenant.
VERDICT_RULES = {
    "PARFAIT": "tous les points clés présents, aucune erreur",
    "PARTIEL": "compréhension globalement correcte, mais incomplète ou avec une imprécision mineure",
    "INCORRECT": "aucun point clé correct, ou une erreur majeure qui contredit un point clé",
}

In [ ]:
def load_fiches(folder: str = FICHES_FOLDER) -> list:
    """Charge toutes les fiches concept d'un dossier.

    Paramètres :
        folder : chemin du dossier qui contient les fichiers .json.

    Retour :
        liste de fiches (dictionnaires), triée par "id".

    Exemple :
        load_fiches("json")[0]["id"]  # "bateau-de-thesee"
    """
    fiches = []
    for path in Path(folder).glob("*.json"):
        fiche = json.loads(path.read_text(encoding="utf-8"))
        for key in REQUIRED_KEYS:
            # une fiche incomplète produirait des dialogues faux sans erreur visible
            if key not in fiche:
                raise ValueError(f"{path.name} : clé manquante « {key} »")
        fiches.append(fiche)
    if not fiches:
        raise ValueError(f"aucune fiche trouvée dans le dossier « {folder} »")
    # ordre fixe : le dataset généré est le même d'une exécution à l'autre
    return sorted(fiches, key=lambda fiche: fiche["id"])


# load_fiches : ordre stable, et chaque question appartient bien à sa fiche.
fiches = load_fiches()
assert [fiche["id"] for fiche in fiches] == sorted(fiche["id"] for fiche in fiches)
assert all(q["id"].startswith(fiche["id"]) for fiche in fiches for q in fiche["questions"])
print(f"load_fiches : {len(fiches)} fiches, {sum(len(fiche['questions']) for fiche in fiches)} questions")

In [ ]:
def write_jsonl(rows: list, path: str) -> int:
    """Écrit une liste d'objets dans un fichier JSONL, un objet par ligne.

    Paramètres :
        rows : liste de dictionnaires (ici : les entrées étiquetées).
        path : chemin du fichier à écrire (exemple : "query_label_llm/monty-hall.jsonl").

    Retour :
        nombre de lignes écrites.

    Exemple :
        write_jsonl(entries, "query_label_llm/monty-hall.jsonl")  # 30
    """
    with open(path, "w", encoding="utf-8") as file:
        for row in rows:
            # ensure_ascii=False garde les accents lisibles pour la relecture à la main
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    return len(rows)


# write_jsonl : le fichier relu ligne par ligne redonne exactement les entrées écrites.
test_rows = [
    {"id": "essai_01", "user_input": "réponse d'essai à accents", "verdict": "PARFAIT"},
    {"id": "essai_02", "user_input": "ligne 1\nligne 2", "verdict": "PARTIEL"},
]
test_path = "write_jsonl_test.jsonl"
lines_count = write_jsonl(test_rows, test_path)
with open(test_path, encoding="utf-8") as file:
    reloaded = [json.loads(line) for line in file]
Path(test_path).unlink()  # fichier d'essai : on ne le laisse pas à côté des vrais fichiers
assert lines_count == 2 and reloaded == test_rows
print(f"write_jsonl : {lines_count} lignes écrites puis relues à l'identique")

In [ ]:
def generate_answers(fiche: dict, question: dict, verdict: str, count: int, llm) -> list:
    """Demande au LLM plusieurs réponses d'apprenant qui méritent toutes le même verdict.

    Paramètres :
        fiche : fiche concept chargée par load_fiches.
        question : une des questions de cette fiche.
        verdict : "PARFAIT", "PARTIEL" ou "INCORRECT".
        count : nombre de réponses demandées.
        llm : fonction qui reçoit un prompt et renvoie un texte (sur Colab : call_llm).

    Retour :
        liste de `count` réponses d'apprenant (textes), toutes différentes.

    Exemple :
        len(generate_answers(fiche, question, "PARTIEL", 4, call_llm))  # 4
    """
    if verdict not in VERDICTS:
        raise ValueError(f"verdict inconnu : {verdict}")
    # seulement les sections placées avant la question : le LLM connaît la mise en scène vue par l'apprenant
    sections_read = fiche["explication"][: question["apres_section"]]
    explication = "\n\n".join(section["contenu"] for section in sections_read)
    points_cles = "\n".join(f"- {point}" for point in question["points_cles"])
    # un seul appel pour toutes les réponses d'un verdict : sans réglage de température,
    # des appels séparés et identiques donneraient des réponses presque pareilles
    prompt = (
        "Tu aides à créer des exemples pour évaluer un tuteur.\n\n"
        f"Concept : {fiche['concept']}\n\n"
        f"Texte lu par l'apprenant :\n{explication}\n\n"
        f"Question posée à l'apprenant : {question['enonce']}\n\n"
        f"Points clés attendus :\n{points_cles}\n\n"
        f"Écris {count} réponses d'apprenants différentes qui méritent toutes le verdict {verdict} "
        f"({VERDICT_RULES[verdict]}). Varie la longueur, le ton et la raison du verdict.\n\n"
        f'Réponds uniquement avec une liste JSON de {count} textes : ["...", "..."].'
    )
    for _ in range(MAX_ATTEMPTS):
        raw = llm(prompt)
        # les LLM entourent souvent le JSON de texte ou de ```json : on garde ce qui est entre crochets
        json_text = raw[raw.find("[") : raw.rfind("]") + 1]
        try:
            answers = [answer.strip() for answer in json.loads(json_text)]
        except (ValueError, TypeError, AttributeError):
            continue  # sortie inutilisable : on redemande
        if len(answers) == count and all(answers) and len(set(answers)) == count:
            return answers
    raise ValueError(f"{question['id']} ({verdict}) : aucune liste valide après {MAX_ATTEMPTS} essais")


# generate_answers : testé avec de faux LLM (liste valide, mauvais nombre, doublons).
first_question = fiches[0]["questions"][0]
answers = generate_answers(fiches[0], first_question, "PARTIEL", 3, lambda prompt: '```json\n["a", "b", "c"]\n```')
assert answers == ["a", "b", "c"]
for bad_reply in ['["a", "b"]', '["a", "a", "b"]', "pas de liste"]:
    try:
        generate_answers(fiches[0], first_question, "PARTIEL", 3, lambda prompt: bad_reply)
        assert False, "une liste invalide doit lever une erreur"
    except ValueError as error:
        last_error = error
print("generate_answers : erreur attendue →", last_error)

In [ ]:
ANSWERS_PER_FICHE = 30


def generate_query_labels(fiche: dict, llm) -> list:
    """Génère les réponses d'apprenant étiquetées d'une fiche, réparties entre ses questions.

    Paramètres :
        fiche : fiche concept chargée par load_fiches.
        llm : fonction qui reçoit un prompt et renvoie un texte (sur Colab : call_llm).

    Retour :
        liste de ANSWERS_PER_FICHE entrées {"id", "concept_id", "question_id", "user_input", "verdict"}.

    Exemple :
        generate_query_labels(fiches[0], call_llm)[0]["id"]  # "bateau-de-thesee_q1_01"
    """
    entries = []
    questions = fiche["questions"]
    for index, question in enumerate(questions):
        # les réponses sont partagées entre les questions ; les premières prennent le reste de la division
        total = ANSWERS_PER_FICHE // len(questions) + (1 if index < ANSWERS_PER_FICHE % len(questions) else 0)
        # environ 30 % PARFAIT, 40 % PARTIEL, 30 % INCORRECT : chaque question reçoit les trois verdicts
        counts = {"PARFAIT": total * 3 // 10, "INCORRECT": total * 3 // 10}
        counts["PARTIEL"] = total - counts["PARFAIT"] - counts["INCORRECT"]
        number = 0
        for verdict in VERDICTS:
            for answer in generate_answers(fiche, question, verdict, counts[verdict], llm):
                number += 1
                entries.append({
                    "id": f"{question['id']}_{number:02d}",
                    "concept_id": fiche["id"],
                    "question_id": question["id"],
                    "user_input": answer,
                    "verdict": verdict,
                })
    return entries


# generate_query_labels : un faux LLM renvoie le nombre de réponses demandé dans le prompt.
def fake_answers_llm(prompt):
    # dernier « Écris » du prompt : certains énoncés de question contiennent aussi ce mot
    count = int(prompt.rsplit("Écris ", 1)[1].split()[0])
    return json.dumps([f"réponse d'essai {number}" for number in range(count)])


for fiche in [fiches[0], fiches[-1]]:  # une fiche à 3 questions, une fiche à 2 questions
    entries = generate_query_labels(fiche, fake_answers_llm)
    verdict_counts = {verdict: sum(entry["verdict"] == verdict for entry in entries) for verdict in VERDICTS}
    assert len(entries) == ANSWERS_PER_FICHE and len({entry["id"] for entry in entries}) == ANSWERS_PER_FICHE
    assert all(entry["concept_id"] == fiche["id"] for entry in entries) and min(verdict_counts.values()) > 0
    print(f"generate_query_labels : {fiche['id']} → {len(entries)} entrées", verdict_counts)
assert entries[0]["id"] == f"{fiches[-1]['questions'][0]['id']}_01"

## Génération avec le LLM de Colab

Cette cellule ne tourne que sur la machine Colab (`google.colab.ai`). `call_llm` se passe en paramètre `llm` à `generate_turn` et `generate_case`.

In [ ]:
from google.colab import ai

# Modèle fixé : le modèle par défaut de Colab peut changer, et les réponses avec lui.
MODEL_NAME = "google/gemini-3.5-flash"


def call_llm(prompt: str) -> str:
    """Envoie un prompt au LLM de Colab et renvoie sa réponse.

    Paramètres :
        prompt : texte envoyé au modèle.

    Retour :
        texte généré par le modèle.

    Exemple :
        call_llm("Réponds uniquement par le mot : ok")  # "ok"
    """
    return ai.generate_text(prompt, model_name=MODEL_NAME)


# Test : le modèle fixé existe et répond par un texte non vide.
models = ai.list_models()
print("Modèles disponibles :", models)
assert MODEL_NAME in models, f"{MODEL_NAME} absent : choisir un modèle de la liste ci-dessus"
reply = call_llm("Réponds uniquement par le mot : ok")
assert isinstance(reply, str) and reply.strip()
print("call_llm :", reply.strip())

In [ ]:
# Génération réelle, sur Colab uniquement : environ 50 appels au LLM pour les six fiches.
# Dossier séparé : les fichiers déjà présents dans query_label ne sont pas remplacés.
OUTPUT_FOLDER = "query_label_llm"

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)
for fiche in fiches:
    entries = generate_query_labels(fiche, call_llm)
    path = f"{OUTPUT_FOLDER}/{fiche['id']}.jsonl"
    verdict_counts = {verdict: sum(entry["verdict"] == verdict for entry in entries) for verdict in VERDICTS}
    print(f"{path} : {write_jsonl(entries, path)} lignes", verdict_counts)